In [14]:
# 강화학습
## 260914

# mastered_bicycle = False

# def try_pedaling():
#     import random
#     return random.choice(["왼발", "오른발"])

# def stayed_balanced():
#     import random
#     return random.random() > 0.3

# def update_strategy(action, reward):
#     pass

# while not mastered_bicycle:
#     action = try_pedaling()
#     balanced = stayed_balanced()

#     if balanced:
#         reward = +10
#         print("Good Job!")

#     else:
#         reward = -5
#         print("Try Again!")

#     update_strategy(action, reward)

#     import random

#     if random.random() > 0.95:
#         mastered_bicycle = True

# print("Success!!")

In [15]:
# class ReinforcementLearningDemo():
#     def __init__(self):
#         self.score = 0
#         self.game_over = False

#     def cartpole_example(self):
#         print("목표: 막대 쓰러뜨리지 않기")

#         while not self.game_over:
#             state = {
#                 "pole_angle": 0.1,
#                 "cart_position": 0.0,
#                 "pole_velocity": 0.02,
#                 "cart_velocity": 0.1
#             }
#             print(state)

#             action = self.choose_action(state)
#             print(f"선택한 행동: {"왼쪽" if action == -1 else "오른쪽"}")
#             reward = self.calculate_reward(state, action)
#             print(f"받은 보상: {reward}")
#             self.update_policy(state, reward)
#             print("전략 업데이트")
#             self.score += reward

#             if self.score < -100:
#                 self.game_over = True

#         print(f"최종 스코어: {self.score}")

#     def choose_action(self, state):
#         import random

#         if random.random() < 0.9:
#             return 1 if state['pole_angle'] > 0 else -1

#         else:
#             return random.choice([-1, 1])

#     def calculate_reward(self, state, action):
#         if abs(state['pole_angle']) > 0.5:
#             return -100

#         else:
#             return +1

#     def update_policy(self, state, reward):
#         learning_rate = 0.1
#         discount_factor = 0.95
#         print("더 나은 전략으로 업데이트")

# demo = ReinforcementLearningDemo()
# demo.cartpole_example()

In [16]:
import gym # 강화학습을 할 환경을 만들 수 있음.
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

from collections import deque

class QNetWork(nn.Module): # MLP랑 기본 모양이 같음
    def __init__(self, state_size, action_size, seed, fc1_units = 64, fc2_units = 64):
        super(QNetWork, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))

        return self.fc3(x)

env = gym.make("CartPole-v1")
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
seed = 42

qnetwork_local = QNetWork(state_size, action_size, seed)
qnetwork_target = QNetWork(state_size, action_size, seed)

optimizer = optim.Adam(qnetwork_local.parameters(), lr = 5e-4)

buffer_size = int(1e5) # 리플레이 버퍼
batch_size = 64
memory = deque(maxlen = buffer_size)

def step(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def sample():
    experiences = random.sample(memory, k = batch_size)
    states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float()
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long()
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences])).float()
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float()
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.uint8)).float()

    return (states, actions, rewards, next_states, dones)

def learn(experiences, gamma):
    states, actions, rewards, next_states, dones = experiences
    Q_target_next = qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
    Q_targets = rewards + (gamma * Q_target_next * (1 - dones))
    Q_expected = qnetwork_local(states).gather(1, actions)
    loss = nn.MSELoss()(Q_expected, Q_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

gamma = 0.99
tau = 1e-3
n_episodes = 200
max_t = 1000

for i_episode in range(1, n_episodes + 1):
    state = env.reset()
    total_reward = 0

    for t in range(max_t):
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)

        with torch.no_grad():
            action_values = qnetwork_local(state_tensor)

        action = np.argmax(action_values.cpu().data.numpy())
        next_state, reward, done, _ = env.step(action)
        step(state, action, reward, next_state, done)
        total_reward += reward

        if len(memory) > batch_size:
            experiences = sample()
            loss = learn(experiences, gamma)
        state = next_state

        if done:
            break

    for target_param, local_param in zip(qnetwork_target.parameters(), qnetwork_local.parameters()):
        target_param.data.copy_(tau * local_param.data + (1 - tau) * target_param)

    print(i_episode, total_reward)

env.close()

c:\Users\KDS-11\Documents\DeepLearning\.venv\Lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\KDS-11\Documents\DeepLearning\.venv\Lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\KDS-11\Documents\DeepLearning\.venv\Lib\site-packages\gym\utils\passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


1 10.0
2 9.0
3 10.0
4 8.0
5 10.0
6 10.0
7 10.0
8 9.0
9 9.0
10 10.0
11 10.0
12 8.0
13 10.0
14 9.0
15 9.0
16 9.0
17 9.0
18 8.0
19 10.0
20 9.0
21 10.0
22 8.0
23 9.0
24 8.0
25 10.0
26 10.0
27 9.0
28 9.0
29 8.0
30 9.0
31 9.0
32 9.0
33 8.0
34 9.0
35 8.0
36 11.0
37 9.0
38 10.0
39 10.0
40 8.0
41 9.0
42 10.0
43 10.0
44 10.0
45 9.0
46 10.0
47 9.0
48 11.0
49 10.0
50 9.0
51 10.0
52 9.0
53 10.0
54 9.0
55 9.0
56 11.0
57 8.0
58 10.0
59 10.0
60 9.0
61 8.0
62 10.0
63 10.0
64 9.0
65 9.0
66 10.0
67 10.0
68 8.0
69 9.0
70 10.0
71 8.0
72 8.0
73 10.0
74 9.0
75 8.0
76 9.0
77 9.0
78 8.0
79 11.0
80 9.0
81 9.0
82 9.0
83 9.0
84 9.0
85 8.0
86 8.0
87 10.0
88 10.0
89 8.0
90 10.0
91 10.0
92 10.0
93 11.0
94 9.0
95 9.0
96 10.0
97 10.0
98 9.0
99 11.0
100 8.0
101 8.0
102 8.0
103 10.0
104 9.0
105 8.0
106 8.0
107 9.0
108 10.0
109 10.0
110 10.0
111 10.0
112 8.0
113 9.0
114 9.0
115 10.0
116 9.0
117 10.0
118 9.0
119 10.0
120 10.0
121 9.0
122 8.0
123 9.0
124 10.0
125 8.0
126 10.0
127 10.0
128 9.0
129 10.0
130 10.0
131 9.0
132 

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_dim = 128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = self.fc2(x)

        return torch.softmax(x, dim = -1)

env = gym.make("CartPole-v1")
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy_net = PolicyNetwork(state_size, action_size)
optimizer = optim.Adam(policy_net.parameters(), lr = 0.01)

def select_action(state):
    state = torch.from_numpy(state).float().unsqueeze(0)
    probs = policy_net(state)
    action = torch.multinomial(probs, num_samples = 1)

    return action.item(), torch.log(probs[0, action.item()])

def reinforce_update(episode_rewards, episode_log_probs, gamma = 0.99):
    R = 0
    returns = []

    for r in episode_rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)

    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    loss = 0

    for log_prob, R in zip(episode_log_probs, returns):
        loss -= log_prob + R

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

num_episodes = 1000

for episode in range(num_episodes):
    state = env.reset()
    episode_rewards = []
    episode_log_probs = []
    done = False

    while not done:
        action, log_prob = select_action(state)
        next_state, reward, done, _ = env.step(action)
        episode_rewards.append(reward)
        episode_log_probs.append(log_prob)
        state = next_state

    loss = reinforce_update(episode_rewards, episode_log_probs)

    if episode % 50 == 0:
        total_reward = sum(episode_rewards)
        print(episode, total_reward, loss)

env.close()

c:\Users\KDS-11\Documents\DeepLearning\.venv\Lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\KDS-11\Documents\DeepLearning\.venv\Lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\KDS-11\Documents\DeepLearning\.venv\Lib\site-packages\gym\utils\passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


0 26.0 18.8145809173584
50 12.0 2.455191135406494
100 11.0 3.4201536178588867
150 9.0 3.5557141304016113
200 18.0 13.080081939697266
250 18.0 10.661155700683594
300 9.0 0.027845025062561035
350 9.0 0.01687335968017578
400 9.0 0.05111873149871826
450 9.0 0.01875472068786621
500 9.0 0.013670086860656738
550 10.0 0.007299184799194336
600 10.0 0.015201568603515625
650 8.0 0.005065202713012695
700 10.0 0.007583022117614746
750 10.0 0.005086183547973633
800 9.0 0.0021647214889526367
850 10.0 0.0026656389236450195
900 10.0 0.0023201704025268555
950 8.0 0.0010753870010375977
